# Model Trainer - Colab GPU

In [ ]:
!pip install kagglehub matplotlib onnx onnxruntime pandas scikit-learn seaborn tensorflow tf2onnx -q

In [ ]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))

## Run training

In [ ]:
# === src/config.py ===
from __future__ import annotations

In [ ]:
import logging
import os
from dataclasses import dataclass, field
from pathlib import Path

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
@dataclass(frozen=True)
class DataConfig:
    baseline_condition: str = "Base"
    amusement_condition: str = "Fun"
    stress_condition: str = "TSST"

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    lstm_units: int = 64
    dropout_rate: float = 0.3
    learning_rate: float = 1e-3
    l2_reg: float = 5e-4
    lstm_dropout: float = 0.2
    lstm_recurrent_dropout: float = 0.0

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    sequence_lengths: tuple[int, ...] = tuple(
        int(item)
        for item in os.environ.get("SEQUENCE_LENGTHS_OVERRIDE", "120,240,360").split(",")
        if item.strip()
    )
    epochs: int = int(os.environ.get("EPOCHS_OVERRIDE", "30"))
    batch_size: int = 16
    export_onnx: bool = True

    patience: int = 15
    reduce_lr_patience: int = 5
    reduce_lr_factor: float = 0.5
    min_lr: float = 1e-6
    positive_class_weight: float = 4.0
    decision_threshold_beta: float = 2.0

    models_dir: Path = Path(os.environ.get("MODELS_DIR_OVERRIDE", "models"))

    def model_dir(self, seq_len: int) -> Path:
        """Return (and create) the output directory for a given sequence length."""
        d = self.models_dir / f"seq_{seq_len}"
        d.mkdir(parents=True, exist_ok=True)
        return d

In [ ]:
DATA = DataConfig()
MODEL = ModelConfig()
TRAINING = TrainingConfig()

In [ ]:
# === src/data/download.py ===
from os import path

In [ ]:
import kagglehub

In [ ]:
def dataset_fetching():
    initial_path = kagglehub.dataset_download("qiriro/stress")
    return path.join(initial_path, "dataset")

In [ ]:
# === src/data/features.py ===
import numpy as np
import pandas as pd

In [ ]:
def filter_hardware_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters the dataset to include only features that a MAX30102 sensor can reasonably extract.
    
    MAX30102 Characteristics:
    - Integrated Pulse Oximetry and Heart-Rate Monitor.
    - Extracts Red and IR (Infrared) PPG signals.
    - Derived data: Heart Rate (HR), SpO2 (Oxygen Saturation), and RR Intervals (RRI).
    - Features below are derived from these raw signals (HRV analysis).
    - Sampling: User specified ~4Hz processed features.
    """
    
    # We select features that can be derived from the heart-rate/RRI signals 
    # provided by the MAX30102 PPG sensor.
    target_features = [
        "MEAN_RR",       # Average time between heartbeats
        "MEDIAN_RR",     # Median time between heartbeats
        "SDRR",          # Standard deviation of RR intervals (Overall HRV)
        "RMSSD",         # Root mean square of successive differences (Parasympathetic activity)
        "SDSD",          # Standard deviation of successive differences
        "SDRR_RMSSD",    # Ratio of SDRR to RMSSD
        "HR",            # Heart Rate (BPM)
        "pNN25",         # Percentage of RR intervals differing by >25ms
        "pNN50",         # Percentage of RR intervals differing by >50ms
        "SD1",           # Poincaré plot short-term variability
        "SD2",           # Poincaré plot long-term variability
    ]
    
    # Identify which columns are present in the current dataframe
    available_features = [col for col in target_features if col in df.columns]
    
    # Return the filtered dataframe with readable names (keeping original case/naming as requested)
    return df[available_features]

In [ ]:
def create_sequences_by_subject(
    subjects_data: list, 
    seq_len: int
) -> tuple[np.ndarray, np.ndarray]:
    """
    Groups data by user and creates sequences with a 10% overlap.
    "No striding" - means we use a fixed shift calculated from the overlap to ensure
    no gaps in the data processing.
    
    Example for seq_len=120:
    Overlap = 12 (10%)
    Shift = 108 (120 - 12)
    Window 1: 0-120
    Window 2: 108-228
    """
    all_X = []
    all_y = []
    
    # Calculate shift based on 10% overlap
    overlap = int(seq_len * 0.1)
    shift = seq_len - overlap
    
    if shift <= 0:
        shift = 1

    for subj_id, features, labels, _ in subjects_data:
        n_samples = len(features)
        
        # Slide through the data using the calculated shift
        for i in range(0, n_samples - 2 * seq_len + 1, shift):
            # Current window (Features)
            x_window = features[i : i + seq_len]
            
            # Next window (Future Target)
            # We look at the labels in the *next* block of time to predict what happens next
            future_labels = labels[i + seq_len : i + 2 * seq_len]
            
            # Majority vote for the future state
            counts = np.bincount(future_labels.astype(np.intp), minlength=3)
            y_label = np.argmax(counts)
            
            all_X.append(x_window)
            all_y.append(y_label)
            
    return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.uint8)

In [ ]:
# === src/models/lstm.py ===
import tensorflow as tf
from tensorflow.keras import Model, layers, regularizers

In [ ]:
def build_rri_lstm(seq_len: int, n_features: int) -> Model:
    inputs = layers.Input(shape=(seq_len, n_features), name="input")
    standardizer = layers.Normalization(axis=-1, name="standardization")

    x = standardizer(inputs)
    x = layers.Bidirectional(
        layers.LSTM(
            MODEL.lstm_units,
            return_sequences=True,
            dropout=MODEL.lstm_dropout,
            recurrent_dropout=MODEL.lstm_recurrent_dropout,
            kernel_regularizer=regularizers.l2(MODEL.l2_reg),
        )
    )(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(MODEL.dropout_rate)(x)
    x = layers.Dense(
        16, activation="relu", kernel_regularizer=regularizers.l2(MODEL.l2_reg)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(MODEL.dropout_rate)(x)

    outputs = layers.Dense(3, activation="softmax", name="stress_state")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=MODEL.learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseCategoricalAccuracy(name="sparse_accuracy"),
        ],
    )
    return model

In [ ]:
# === src/data/load.py ===
import os
from collections.abc import Generator

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def _parse_quest(filepath: str):
    df = pd.read_csv(filepath, header=None)
    order_str = str(df.iloc[1, 0]).removeprefix("# ORDER;")
    start_str = str(df.iloc[2, 0]).removeprefix("# START;")
    end_str = str(df.iloc[3, 0]).removeprefix("# END;")
    conditions = order_str.split(";")
    starts = [float(v) for v in start_str.split(";") if v]
    ends = [float(v) for v in end_str.split(";") if v]
    return list(zip(conditions[: len(starts)], starts, ends))

In [ ]:
def _label_full_timeline(
    times_min: np.ndarray, schedule: list[tuple[str, float, float]]
) -> np.ndarray:
    if not schedule:
        return np.zeros(times_min.shape, dtype=np.intp)

    labels = np.full(len(times_min), -1, dtype=np.intp)
    for condition, start, end in schedule:
        normalized_condition = condition.strip().lower()
        if normalized_condition == DATA.stress_condition.lower():
            mask = (times_min >= start) & (times_min <= end)
            labels[mask] = 2
        elif normalized_condition == DATA.amusement_condition.lower():
            mask = (times_min >= start) & (times_min <= end)
            labels[mask] = 1
        elif normalized_condition == DATA.baseline_condition.lower():
            mask = (times_min >= start) & (times_min <= end)
            labels[mask] = 0
    return labels

In [ ]:
def load_subject_hrv(subj_id: int, ds_path: str):
    """
    Loads processed HRV chest data for a subject and filters for hardware capabilities.
    """
    hrv_path = os.path.join(
        ds_path, "1. processed", "hrv", "wesad", "raw", "chest", f"S{subj_id}.xlsx"
    )
    quest_path = os.path.join(
        ds_path, "0. interim", "wesad", "Labels", f"S{subj_id}_quest.csv"
    )

    # 1. Load data
    df = pd.read_excel(hrv_path)
    
    # 2. Extract Time for labeling
    times_min = df["Time"].values
    
    # 3. Filter hardware specific features using the DataFrame directly
    df_filtered = filter_hardware_features(df)
    features = df_filtered.values.astype(np.float32)
    feature_names = df_filtered.columns.tolist()

    # 4. Generate Labels
    schedule = _parse_quest(quest_path)
    labels = _label_full_timeline(times_min, schedule)
    
    # 5. Filter out invalid/unlabeled segments
    valid = labels >= 0
    features = features[valid]
    labels = labels[valid]

    return features, labels, feature_names

In [ ]:
def load_all_subjects_hrv() -> Generator[tuple[int, np.ndarray, np.ndarray, list[str]]]:
    """
    Yields data for all available subjects.
    """
    ds_path = dataset_fetching()
    hrv_dir = os.path.join(ds_path, "1. processed", "hrv", "wesad", "raw", "chest")
    available = sorted(
        int(f.removeprefix("S").removesuffix(".xlsx")) for f in os.listdir(hrv_dir)
    )
    for subj_id in available:
        yield subj_id, *load_subject_hrv(subj_id, ds_path)

In [ ]:
# === src/training/export.py ===
from pathlib import Path

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model

In [ ]:
def export_to_onnx(model: Model, seq_len: int, n_features: int, model_dir: Path):
    import tf2onnx

    onnx_path = model_dir / "model.onnx"
    input_name = "input"

    if hasattr(model, "input_names") and model.input_names:
        input_name = model.input_names[0]

    spec = (
        tf.TensorSpec((None, seq_len, n_features), tf.float32, name=input_name),
    )

    try:
        model_proto, _ = tf2onnx.convert.from_keras(
            model,
            input_signature=spec,
            opset=13,
            output_path=str(onnx_path),
        )
        print(f"Successfully converted to ONNX: {onnx_path}")
    except Exception as e:
        print(
            f"Direct ONNX conversion failed: {e}. Trying via temporary SavedModel..."
        )
        import subprocess
        import sys
        import tempfile

        with tempfile.TemporaryDirectory() as temp_dir:
            temp_saved_model = str(Path(temp_dir) / "temp_save")
            tf.saved_model.save(model, temp_saved_model)

            cmd = [
                sys.executable,
                "-m",
                "tf2onnx.convert",
                "--saved-model",
                temp_saved_model,
                "--output",
                str(onnx_path),
                "--opset",
                "13",
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(
                    f"Successfully converted to ONNX via temporary SavedModel: {onnx_path}"
                )
            else:
                print(f"ONNX conversion CLI fallback failed: {result.stderr}")

In [ ]:
# === src/training/trainer.py ===
from __future__ import annotations

In [ ]:
import logging
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
def _validate_inputs(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    seq_len: int,
) -> None:
    if X_train.ndim != 3:
        raise ValueError(
            f"X_train must be 3-D (samples, timesteps, features), got {X_train.shape}"
        )
    if X_train.shape[1] != seq_len:
        raise ValueError(
            f"X_train timesteps ({X_train.shape[1]}) != seq_len ({seq_len})"
        )
    if X_train.shape[0] != y_train.shape[0]:
        raise ValueError(
            f"X_train / y_train sample count mismatch: {X_train.shape[0]} vs {y_train.shape[0]}"
        )

In [ ]:
def _build_callbacks() -> list[tf.keras.callbacks.Callback]:
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=TRAINING.patience,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=TRAINING.reduce_lr_factor,
            patience=TRAINING.reduce_lr_patience,
            min_lr=TRAINING.min_lr,
            verbose=1,
        ),
    ]

In [ ]:
def _save_training_curves(
    history: tf.keras.callbacks.History,
    model_dir: Path,
) -> None:
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(history.history["loss"], label="train")
        ax1.plot(history.history["val_loss"], label="val")
        ax1.set_title("Loss")
        ax1.set_xlabel("Epoch")
        ax1.legend()

        ax2.plot(history.history["accuracy"], label="train")
        ax2.plot(history.history["val_accuracy"], label="val")
        ax2.set_title("Accuracy")
        ax2.set_xlabel("Epoch")
        ax2.legend()

        plt.tight_layout()
        out = model_dir / "training_curves.png"
        plt.savefig(out, dpi=150)
    except Exception:
        logger.exception("Failed to save training curves.")
    finally:
        plt.close("all")

In [ ]:
def _save_model(model: Model, model_dir: Path) -> None:
    out = model_dir / "model.keras"
    model.save(str(out))

In [ ]:
def train_model(
    seq_len: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
) -> tuple[Model, tf.keras.callbacks.History]:

    _validate_inputs(X_train, y_train, X_val, y_val, seq_len)

    n_features = X_train.shape[2]
    model_dir = TRAINING.model_dir(seq_len)

    logger.info(
        "Training seq_len=%d | samples=%d | features=%d",
        seq_len,
        len(X_train),
        n_features,
    )

    # 1. Build Model
    model = build_rri_lstm(seq_len, n_features)
    
    # 2. Adapt Normalization Layer (Keras layer handles standardization)
    standardization_layer = model.get_layer("standardization")
    print("Adapting Keras Normalization layer to training data...")
    standardization_layer.adapt(X_train)
    
    # 3. Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=MODEL.learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseCategoricalAccuracy(name="sparse_accuracy"),
        ],
    )

    # 4. Prepare Datasets
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_ds = train_ds.shuffle(len(X_train)).batch(TRAINING.batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_ds = val_ds.batch(TRAINING.batch_size).prefetch(tf.data.AUTOTUNE)

    # 5. Fit
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TRAINING.epochs,
        callbacks=_build_callbacks(),
        verbose=1,
    )

    # 6. Save Artifacts
    _save_training_curves(history, model_dir)
    if TRAINING.export_onnx:
        export_to_onnx(model, seq_len, n_features, model_dir)
    _save_model(model, model_dir)

    return model, history

In [ ]:
# === src/evaluate/metrics.py ===
import json

In [ ]:
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
def _summarize_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> dict[str, float]:
    y_pred = np.argmax(y_prob, axis=1)
    score = f1_score(y_true, y_pred, average="macro", zero_division=0)
    best_metrics = {
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "fbeta": float(score),
    }
    return best_metrics

In [ ]:
def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray, seq_len: int):
    eval_result = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
    y_prob = model.predict(X_test, verbose=0)
    threshold_metrics = _summarize_predictions(y_test, y_prob)
    y_pred = np.argmax(y_prob, axis=1)

    acc = float(eval_result.get("accuracy", accuracy_score(y_test, y_pred)))
    test_loss = float(eval_result.get("loss", 0.0))
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    class_names = ["baseline", "amusement", "stress"]
    actual_counts = np.bincount(y_test.astype(np.intp), minlength=3)
    predicted_counts = np.bincount(y_pred.astype(np.intp), minlength=3)

    metrics = {
        "seq_len": seq_len,
        "threshold_precision": float(threshold_metrics["precision"]),
        "threshold_recall": float(threshold_metrics["recall"]),
        "threshold_fbeta": float(threshold_metrics["fbeta"]),
        "accuracy": float(acc),
        "test_loss": float(test_loss),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "mean_baseline_probability": float(np.mean(y_prob[:, 0])),
        "mean_amusement_probability": float(np.mean(y_prob[:, 1])),
        "mean_stress_probability": float(np.mean(y_prob[:, 2])),
    }

    print(f"\nTest loss: {test_loss:.4f}")
    print(f"Test accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")
    print(
        "Mean class probabilities - "
        f"baseline: {metrics['mean_baseline_probability']:.4f}, "
        f"amusement: {metrics['mean_amusement_probability']:.4f}, "
        f"stress: {metrics['mean_stress_probability']:.4f}"
    )
    print(
        "Class counts - "
        f"actual baseline={actual_counts[0]}, amusement={actual_counts[1]}, stress={actual_counts[2]} | "
        f"pred baseline={predicted_counts[0]}, amusement={predicted_counts[1]}, stress={predicted_counts[2]}"
    )
    print(f"Sample probabilities: {np.array2string(y_prob[:10], precision=4)}")

    model_dir = TRAINING.models_dir / f"seq_{seq_len}"
    with open(model_dir / "results.json", "w") as f:
        json.dump(metrics, f, indent=4)

    report = classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=class_names,
        zero_division=0,
    )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(f"Confusion Matrix (seq_len={seq_len})")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")

    plot_path = model_dir / "confusion_matrix.png"
    plt.savefig(plot_path)
    plt.close()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(["test_accuracy", "test_loss"], [acc, test_loss], color=["#2b8a3e", "#c92a2a"])
    ax.set_ylim(0.0, max(1.0, test_loss * 1.15, acc * 1.15))
    ax.set_title(f"Test Metrics (seq_len={seq_len})")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "test_metrics_summary.png")
    plt.close()

    return report

In [ ]:
def plot_correlations(features: np.ndarray, feature_names: list[str]):
    TRAINING.models_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(features, columns=feature_names)
    plt.figure(figsize=(10, 8))
    sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Feature Correlation Map")
    plt.savefig(TRAINING.models_dir / "feature_correlation.png")
    plt.close()

In [ ]:
def plot_performance_summary(results: list[dict[str, float | int]]) -> None:
    if not results:
        return

    model_dir = TRAINING.models_dir
    model_dir.mkdir(parents=True, exist_ok=True)

    seq_lens = [int(item["seq_len"]) for item in results]
    x = np.arange(len(seq_lens))
    width = 0.18

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - 1.5 * width, [float(item["accuracy"]) for item in results], width, label="Accuracy")
    ax.bar(x - 0.5 * width, [float(item["precision"]) for item in results], width, label="Precision")
    ax.bar(x + 0.5 * width, [float(item["recall"]) for item in results], width, label="Recall")
    ax.bar(x + 1.5 * width, [float(item["f1"]) for item in results], width, label="F1")

    ax.set_xticks(x)
    ax.set_xticklabels([f"seq_{seq}" for seq in seq_lens])
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Model Performance by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "performance_summary.png")
    plt.close()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(
        x - 1.0 * width,
        [float(item["mean_baseline_probability"]) for item in results],
        width,
        label="Baseline",
    )
    ax.bar(
        x,
        [float(item["mean_amusement_probability"]) for item in results],
        width,
        label="Amusement",
    )
    ax.bar(
        x + 1.0 * width,
        [float(item["mean_stress_probability"]) for item in results],
        width,
        label="Stress",
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"seq_{seq}" for seq in seq_lens])
    ax.set_ylabel("Mean Probability")
    ax.set_title("Mean Class Probabilities by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "class_probability_summary.png")
    plt.close()

In [ ]:
def plot_modal_timing(run_log: list[dict[str, float | int]]) -> None:
    if not run_log:
        return

    model_dir = TRAINING.models_dir
    model_dir.mkdir(parents=True, exist_ok=True)

    seq_lens = [int(item["seq_len"]) for item in run_log]
    durations = [float(item["duration_seconds"]) for item in run_log]
    wait_times = [float(item["wait_seconds"]) for item in run_log]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([f"seq_{seq}" for seq in seq_lens], durations, label="Training duration")
    ax.plot([f"seq_{seq}" for seq in seq_lens], wait_times, marker="o", label="Queue-to-finish wait")
    ax.set_ylabel("Seconds")
    ax.set_title("Modal Run Timing by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "modal_timing_summary.png")
    plt.close()

In [ ]:
# === main.py ===
import os
import sys
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
# Ensure encoding is set for Windows output
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

In [ ]:
def run() -> None:
    """
    Main entry point for the model trainer.
    Executes Data Loading, Feature Engineering, Sequence Creation, and Training.
    """
    print("--- Starting ZenDoc Model Trainer ---")
    
    # 1. Load Data
    print("Loading processed HRV data for all subjects...")
    subjects = list(load_all_subjects())
    print(f"Successfully loaded {len(subjects)} subjects.")

    if len(subjects) == 0:
        print("Error: No subjects found. Check dataset path.")
        return

    # Extract feature names from the first subject
    feature_names = subjects[0][3]
    n_features = len(feature_names)
    print(f"Features for MAX30102 ({n_features}): {feature_names}")

    # 2. Process each sequence length defined in config
    for seq_len in TRAINING.sequence_lengths:
        print(f"\n{'='*60}")
        print(f"Processing Sequence Length: {seq_len}")
        print(f"{'='*60}")

        # 3. Create Sequences grouped by subject (Subject-aware sequence creation)
        print(f"Creating temporal sequences (Window: {seq_len})...")
        X_all, y_all = create_sequences_by_subject(subjects, seq_len)
        print(f"Total sequences created: {len(X_all)}")

        # 4. Stratified Split (80% Train, 10% Val, 10% Test)
        # We pool all subjects to create a general AI model.
        print("Performing stratified data split (Train/Val/Test)...")
        X_train, X_temp, y_train, y_temp = train_test_split(
            X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
        )

        # 5. Model Training
        # Note: Normalization is handled within the Keras model layer.
        print(f"Training parameters - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        print(f"Stress ratios - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")
        
        model, _history = train_model(seq_len, X_train, y_train, X_val, y_val)

        # 6. Evaluation
        print(f"\nEvaluating model for seq_len={seq_len}...")
        report = evaluate_model(model, X_test, y_test, seq_len)
        print(f"Classification Report:\n{report}")

    print(f"\nAll models and artifacts saved to: {TRAINING.models_dir}")
    print("Architecture: [MAX30102 HRV Features] -> LSTM Sequence -> Future Stress Prediction")

In [ ]:
if __name__ == "__main__":
    run()

## Save models to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, glob

out_dir_name = "colab/model-trainer/models"
output_path = f"/content/drive/MyDrive/{out_dir_name}"

# Source directory for local models
local_models_dir = "models"

# Remove existing directory in Drive if it exists to ensure a clean copy
if os.path.exists(output_path):
    shutil.rmtree(output_path)

# Copy all contents from the local models directory to Google Drive
for src_path in glob.glob(f"{local_models_dir}/**", recursive=True):
    if os.path.isfile(src_path):
        # Construct the destination path, preserving subdirectory structure
        dst_path = src_path.replace(local_models_dir, output_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(src_path, dst_path)
print(f"Saved models to {output_path}")
